In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dev = pd.read_csv("../data/casas_dev.csv")

1.1

Fragmento random del dataset

In [ ]:
dev.sample(5)

Variables posibles dentro de unidades y tipo

In [ ]:
print(dev['tipo'].unique())
print(dev['unidades'].unique())


Lo uso porque de esta manera me aseguro que no haya otros valores posibles para luego categorizar.

- Columna unidades: deberia estar en una misma medida, elijo m2
- Columna pisos: demasiados NaN
- Columna pileta: convertir en 1(True) y 0(False)
- Columna tipo: pasar de variable categorica a numero
- Normalizar las variables (las escalas son demasiado distintas)

Busco NaNs en cada columna:

In [ ]:
columnas = []
porcentajes = []
for col in dev.columns:
	columna = dev[col]
	total_NaNs_en_columna = columna.isna().sum() #sumo cantidad de NaNs 
	porcentaje_NaNs_en_columna = (total_NaNs_en_columna/dev.shape[0]) * 100

	columnas.append(col)
	porcentajes.append(porcentaje_NaNs_en_columna)

plt.figure()
plt.bar(columnas, porcentajes)
plt.xticks(rotation=45)
plt.ylabel("Porcentaje de NaNs")
plt.title("Porcentaje de valores faltantes por columna")
plt.tight_layout()
plt.show()


Para los NaNs de edad usare la media de esa feature para reemplazarlos.
Para los de precio hare lo mismo.
En ambos casos calculare la media contemplando su ubicacion. (esto lo hare mas adelante porque primero debo separar en train y validation para no generar data l)

Decido eliminar la columna de pisos porque hay mas de un 50% de valores faltantes, lo que puede producir sesgos importantes. 

Las demas no tienen  porcentajes tan significativos 
--> utilizo mediana para estimar valores numericos


Antes de eliminar veo la correlacion de precios con pisos (en los valores que no son NaN), para tomar desicion final

In [ ]:
correlacion_pisos_precios = dev['precio'].corr(dev['pisos'])
print(correlacion_pisos_precios)

Efectivamente es conveniente eliminar la columna de pisos. Ademas de los valores faltantes, la correlacion entre precios y pisos (con los valores que si existe) es muy debil.

In [ ]:
dev = dev.drop('pisos', axis=1) #elimina columna pisos

Identico como estan distribuidas las casas en lat y long

In [ ]:
plt.figure()

lat = dev['lat']
lon = dev['lon']
plt.scatter(lat,lon,s=50, alpha=0.3)
plt.xlabel("lat")
plt.ylabel("lon")
plt.title("Distribucion latitud y longitud")

plt.show()

Se puede ver que la ubicacion de las casas (en lat y lon) estan concentradas unicamente en dos lugares:
Mirando el grafico se puede ver que estas son: 1 --> (-34, -58): buscandolo en el mapa equivale a la provincia de Buenos Aires (Argentina) y 2 --> (40,-74): buscandolo en el mapa equivale a New York (USA)

Agrego Feature nueva Ubicacion que reemplaza lat y long. 
Los valores seran 1 si la ubicacion es New York y 0 si es Buenos Aires (para no usar variables categoricas)

In [ ]:
dev['Ubicacion'] = (dev['lat'] > 0).astype(int) #crea una columna nueva fijandose por la latitud si es New York o Argentina (solo hace falta ver latitud)

Elimino lat y long porque ubicacion las reemplazo

In [ ]:
dev = dev.drop('lat',axis=1)
dev = dev.drop('lon',axis=1)

Cambio columna pileta a 1/0 en vez de True/False

In [ ]:
dev['pileta'] = (dev['pileta']).astype(int)

Convierto sqft en m2
Cuando la variable en Area/metros cubiertos este escrita en sqft, la paso a metros

Cuando la variable en Area/metros cubiertos este escrita en sqft, la paso a metros

In [ ]:
dev.loc[dev['Ubicacion'] == 1, 'Área'] = dev.loc[dev['Ubicacion'] == 1, 'Área'] / 10.764 
dev.loc[dev['Ubicacion'] == 1, 'metros_cubiertos'] = dev.loc[dev['Ubicacion'] == 1, 'metros_cubiertos'] / 10.764 

Ahora elimino la columna unidades porque ya esta unificado en m2

In [ ]:
dev = dev.drop('unidades',axis=1)

Tengo que cambiar tipo a variables numericas porque es categorica:
Uso One-Hot Encodding

In [ ]:
dev['es_casa'] = (dev['tipo'] == 'casa').astype(int)
dev['es_ph'] = (dev['tipo'] == 'ph').astype(int)

Si no es casa ni ph, es dept

Elimino tipo porque no me sirve

In [ ]:
dev = dev.drop('tipo', axis=1)

Analizo rango de precios en Argentina

In [ ]:
print(dev.loc[dev['Ubicacion'] == 0, 'precio'].describe())

Analizo rango de precios en USA

In [ ]:
print(dev.loc[dev['Ubicacion'] == 1, 'precio'].describe())

Tomo en cuenta que hay algo raro en los precios de Argentina

CHECK DE CAMBIOS EN EL DATASET

In [ ]:
dev.sample(5)

1.2

Analisis con Boxplot

In [ ]:
plt.figure()
plt.boxplot(dev.loc[dev['Ubicacion'] == 0, 'precio'].dropna())
plt.title('Boxplot Precio - Buenos Aires')
plt.show()

plt.figure()
plt.boxplot(dev.loc[dev['Ubicacion'] == 1, 'precio'].dropna())
plt.title('Boxplot Precio - Nueva York')
plt.show()
for col in dev.select_dtypes(include=['number']).columns:
	if col != 'precio' and col != 'Ubicacion' and col != 'pileta' and col != 'es_casa' and col != 'es_ph':
		plt.figure()
		dev.boxplot(column=col)
		plt.title(f'Boxplot de {col}')
		plt.show()

In [ ]:
dev['pileta'].value_counts(normalize=True)*100

Un 82% de casas no tiene pileta pero hay un 18% que si. La feature puede aportar algo al modelo.

Analisis Boxplots (tomando en cuenta lo importante): 

- Precios: 
Buenos Aires: Concentrados entre 0 y 20000 con outliers arriba de 80000. Sesgo hacia valores altos.
Nueva York: Concentrados entre 100000 y 250000 con outliers arriba de 400000 y cercanos a 0.

- Area:
Concentrada entre 50 y 200 con outliers arriba de 250.

- Metros Cubiertos:
Concentrado entre 75 y 125 pero con outliers arriba de 200

- Ambientes:
Rango entre 1 y 10. Distribucion pareja.

- Edad:
Valores concentrados entre 0 y 40 años, otliers arriba de 80 años, generando sesgo hacia casas nuevas.



Los outliers pueden distorsionar el modelo. 
Los precios entre Buenos Aires y Nueva York estan en escalas muy distintas. Ademas, en cada uno hay valores muy cercanos a 0. Los que sean 0 seran detectados como error y los que no habria que analizar para detectar si pueden ser alquileres.
La normalizacion lleva todas las variables a una misma escala, permitiendo que el algoritmo de gradiente descendiente converja de forma mas rapida y estable.


In [ ]:
for col in dev.select_dtypes(include=['number']).columns:
    if col != 'precio' and col != 'es_casa' and col != 'es_ph':
        plt.figure()
        sns.scatterplot(data=dev, x=col, y='precio', hue='Ubicacion')
        plt.title(f'precio vs {col}')
        plt.show()

Relaciones importantes:
- Precio vs Area/Metro cubiertos:
Cuanto mas terreno/espacio --> mas alto el precio

- Precio vs Pileta:
En Buenos Aires aumenta el precio al tener pileta (por clima)

-Precio vs Ubicacion:
Claramente hay diferencias de escala por la moneda.


- Precio vs ambientes: 
Telación poco clara

- Precio vs edad: 
No depende tanto


1.3

Separo dataset (80% train - 20% validation)

In [ ]:
dev_shuffled = dev.sample(frac=1, random_state=42) 

Creo otra variable para dev, para no cambiar memoria de dev (variable original)

Uso random_state para que siempre se mezcle igual y comparar resultados

In [ ]:
total_filas = dev_shuffled.shape[0]
q_filas_train = int(0.8*total_filas)
train_set = dev_shuffled[:q_filas_train]
validation_set = dev_shuffled[q_filas_train:]

train_set.to_csv("../data/casas_train.csv", index=False)
validation_set.to_csv("../data/casas_validation.csv", index=False)

Normalizacion de variables

Normalizo en memoria (no cambio csv) --> Quiero mantener dataset original

In [ ]:
train = pd.read_csv("../data/casas_train.csv")
validation = pd.read_csv("../data/casas_validation.csv")

Reemplazo de NaNs

In [ ]:
media_precios_NYC = (train.loc[train['Ubicacion'] == 1, 'precio']).mean()
media_precios_BUE = (train.loc[train['Ubicacion'] == 0, 'precio']).mean()
media_edad_NYC = (train.loc[train['Ubicacion'] == 1, 'edad']).mean()
media_edad_BUE = (train.loc[train['Ubicacion'] == 0, 'edad']).mean()

for parte in [train,validation]:
	parte.loc[(parte['precio'].isna()) & (parte['Ubicacion'] == 1), 'precio'] = media_precios_NYC
	parte.loc[(parte['precio'].isna()) & (parte['Ubicacion'] == 0), 'precio'] = media_precios_BUE
	parte.loc[(parte['edad'].isna()) & (parte['Ubicacion'] == 1), 'edad'] = media_edad_NYC
	parte.loc[(parte['edad'].isna()) & (parte['Ubicacion'] == 0), 'edad'] = media_edad_BUE

In [ ]:
media_var = {}
for col in train.columns:
	if col != 'Ubicacion' and col != 'pileta' and col != 'es_casa' and col != 'es_ph':
		media_var[col] = (train[col].mean(), train[col].var())


Consigo media y varianza para poder normalizar
--> Formula de normalizacion a usar = (x - media) / desvio

In [ ]:
for parte in [train,validation]:
	for col in parte.columns:
		if col != 'Ubicacion' and col != 'pileta' and col != 'es_casa' and col != 'es_ph':
			media,var = media_var[col]
			parte[col] = (parte[col] - media) / np.sqrt(var) #modifica el dataframe



Visualizacion de variables normalizadas:

In [ ]:
train.sample(5)

In [ ]:
validation.sample(5)

In [ ]:
pd.set_option('display.float_format', '{:.4f}'.format)
print("Antes de normalizar:")
print(pd.read_csv("../data/casas_train.csv").describe())
print("\nDespués de normalizar:")
print(train.describe())

Todo se normalizo correctamente